# Cruce de mapas de riesgos con controles de ISOLUCION

### Importar librerías necesarias

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz

### Limpiar datos

#### Mapa de riesgos consolidado (desde Excel)

##### Cargar data de mapa de riesgos consolidado

In [2]:
# Leer archivo
mapa_excel = pd.read_excel('Consolidado_2025_v13.xlsx', sheet_name = 'Mapa Riesgos')

/home/jairoescrito/miniconda3/envs/jairoescrito/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


##### Limpiar data de mapa de riesgos consolidado

In [3]:
# Seleccionar variables de inetrés
mapa_ini = mapa_excel.iloc[9:,[0,2,3,6,9,10,11,13,14,15,29,30,31,32,33]]
# Eliminar filas en blanco
mapa_ini.dropna(how='all', inplace=True)
# Resetear índices 
mapa_ini = mapa_ini.reset_index(drop=True)
# Eliminar filas definitivas
mapa_ini = mapa_ini.iloc[:-3, :]

In [4]:
# Extraer nombres de columnas
columnas_mapa_ini = mapa_ini.iloc[0,:]
columnas_mapa_ini = columnas_mapa_ini.reset_index().iloc[:,1]
# Ajustar texto
columnas_mapa_ini = (columnas_mapa_ini.astype(str)
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("\n", ""))

In [5]:
# Cambiar algunos textos
columnas_mapa_ini.iloc[5] = "%_probabilidad_inherente"
columnas_mapa_ini.iloc[8] = "%_impacto_inherente"
columnas_mapa_ini.iloc[11] = "%_probabilidad_residual"
columnas_mapa_ini.iloc[13] = "%_impacto_residual"

In [6]:
# Cambiar los nombres de las columnas
mapa_ini.columns = columnas_mapa_ini
mapa_ini = mapa_ini.iloc[1:,:]

In [7]:
# Eliminar caracteres de espacios en blanco que están de mas
# Seleccionar las columnas que son tipo object (texto)
cols_texto = mapa_ini.select_dtypes(include='object').columns
# Para cada columna reemplazar el \n por espacio y luego eliminar espacios al inicio y al final 
mapa_ini[cols_texto] = (mapa_ini[cols_texto]
    .apply(lambda col: col.where(
        col.isna(),
        col.astype(str).str.replace(r'\\n|\n', ' ', regex=True).str.strip()
    ))
)

In [8]:
# Llenar los NaN hacia abajo
mapa_ini = mapa_ini.ffill()
# Cada riesgo se repite varias veces (segun la cantidad de controles)
# La última fila de cada riesgo tiene el valor final de riesgo residual
# Se selecciona de cada grupo 
mapa_ini = mapa_ini.groupby('consecutivo', sort=False).last().reset_index()

In [9]:
# Convertir a números las columnas que si son numéricas
for col in mapa_ini.select_dtypes(include='object').columns:
    mapa_ini[col] = pd.to_numeric(mapa_ini[col], errors='coerce').fillna(mapa_ini[col])

#### Mapa de riesgos desde controles ISOLUCION

##### Cargar data de reporte de controles ISOLUCION

In [10]:
# Leer archivo
mapa_iso =  pd.read_excel('Reporte Mejoramiento Continuo - Controles.xlsx', 
                        sheet_name = 'Data')

##### Limpiar data de reporte de riesgos

In [11]:
# Seleccionar solo las columnas relativas a los riesgos
mapa_rep = mapa_iso.iloc[:, [1, 2, 7, 9]].copy()

In [12]:
# Eliminar las filas en blanco
mapa_rep.dropna(how='all', inplace=True)
# Resetear índices 
mapa_rep = mapa_rep.reset_index(drop=True)

In [13]:
# Eliminar mayúsculas y acentos en los nombres de las columnas
mapa_rep.columns = (mapa_rep.columns
                    .str.lower()
                    .str.normalize('NFKD')
                    .str.encode('ascii', errors='ignore')
                    .str.decode('utf-8'))

In [14]:
# Cambiar número de identificación del riesgo a entero
mapa_rep['num'] = mapa_rep['num'].astype(int)

## Cruce de riesgos identificados

In [15]:
# Función para limpiar texto y poder hacer la comparación del texto del riesgo en las dos tablas
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    # Minúsculas
    texto = texto.lower()
    # Quitar tildes
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                    if unicodedata.category(c) != 'Mn')
    # Quitar caracteres especiales
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    # Quitar espacios dobles
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [16]:
# Limpiar el texto de cada "texto de riesgo" en cada tabla (la que viene de los mapas y la que viene de ISOLUCION)
mapa_ini['desc_limpia'] = mapa_ini['descripción_del_riesgo'].apply(limpiar_texto)
mapa_rep['desc_limpia'] = mapa_rep['descripcion'].apply(limpiar_texto)

In [1]:
# Cruce de textos de riesgo para evaluar coincidencia de textos
# Los textos que se cargaron a ISOLUCION en algunos casos son diferentes a los que están en los mapas
# Les adicionaron texto, les cambiaron la ortografía, etc
# Obtener top N candidatos para cada riesgo de mapa_ini
N = 5
choices = mapa_rep['desc_limpia'].tolist()

candidatos_todos = []
for idx, row in mapa_ini.iterrows():
    matches = process.extract(
        row['desc_limpia'],
        choices,
        scorer=fuzz.token_sort_ratio,
        limit=N
    )
    for match in matches:
        candidatos_todos.append({
            'consecutivo_ini': row['consecutivo'],
            'proceso': row['proceso'],
            'desc_ini': row['descripción_del_riesgo'],
            'match_idx': match[2],
            'num_rep': mapa_rep.iloc[match[2]]['num'],
            'proceso_rep': mapa_rep.iloc[match[2]]['proceso'],
            'desc_rep': mapa_rep.iloc[match[2]]['descripcion'],
            'score_match': match[1]
        })

# Ordenar por score descendente y asignar sin repetir num_rep
df_candidatos = pd.DataFrame(candidatos_todos).sort_values('score_match', ascending=False)

asignados_rep = set()
asignados_ini = set()
resultados_final = []

for _, row in df_candidatos.iterrows():
    if row['consecutivo_ini'] not in asignados_ini and row['num_rep'] not in asignados_rep:
        row['estado_match'] = 'automatico' if row['score_match'] >= 85 \
                             else 'revision' if row['score_match'] >= 70 \
                             else 'sin_match'
        resultados_final.append(row)
        asignados_ini.add(row['consecutivo_ini'])
        asignados_rep.add(row['num_rep'])

df_match = pd.DataFrame(resultados_final).sort_values('consecutivo_ini')
print(df_match['estado_match'].value_counts())

NameError: name 'mapa_rep' is not defined

In [25]:
df_revision = df_match[df_match['estado_match'] != 'automatico'][
    ['consecutivo_ini', 'proceso', 'desc_ini', 'proceso_rep', 'desc_rep', 'score_match', 'estado_match']
].sort_values('score_match', ascending=False)

print(df_revision.iloc[0,1])  # desc_ini
print(df_revision.iloc[0,3])  # desc_rep

DIRECCIONAMIENTO ESTRATÉGICO Y ARTICULACIÓN TERRITORIAL
Fomento de La Ciencia, Tecnología e Innovación


In [28]:
duplicados = df_match[df_match['estado_match'] == 'automatico']\
    .groupby('num_rep')\
    .filter(lambda x: len(x) > 1)\
    [['consecutivo_ini', 'proceso', 'num_rep', 'desc_ini', 'proceso_rep', 'desc_rep', 'score_match']]\
    .sort_values('num_rep')

print(f"Matches duplicados: {len(duplicados)}")
duplicados

Matches duplicados: 4


,consecutivo_ini,proceso,num_rep,desc_ini,proceso_rep,desc_rep,score_match
92,93,GESTIÓN DE LA MOVILIDAD CONTEMPORANEA,5470,Posibilidad de afectaciòn economica y reputaci...,Gestion de la Movilidad Contemporánea,Posibilidad de afectaciòn economica y reputaci...,100.000000
91,92,GESTIÓN DE LA MOVILIDAD CONTEMPORANEA,5470,Posibilidad de afectaciòn economica y reputaci...,Gestion de la Movilidad Contemporánea,Posibilidad de afectaciòn economica y reputaci...,88.617886
27,28,PROMOCIÓN DEL DESARROLLO EDUCATIVO,5486,Posibilidad de afectación administrativa y fin...,Promoción del Desarrollo Educativo,Posibilidad de afectación económica y reputaci...,85.608856
26,27,PROMOCIÓN DEL DESARROLLO EDUCATIVO,5486,Posibilidad de afectación administrativa y fin...,Promoción del Desarrollo Educativo,Posibilidad de afectación económica y reputaci...,93.702771
